# FoundML 3 — Week 2 Coding Assignment (Student Version)
## Ensemble Learning (Real Dataset): Voting and Stacking on UCI Letter Recognition

In this assignment you will demonstrate the value of **combining diverse learners** using:

1. **Single models** (Logistic Regression, Gaussian Naive Bayes, Decision Tree)
2. **Soft Voting** (average predicted probabilities)
3. **Stacking** (a meta-learner learns how to combine base learners)

Dataset: **UCI Letter Recognition** (26-class classification, engineered numeric features).  
Main metric: **Macro F1** (treats all classes equally).

Difficulty level: **moderate**.

All numerical answers must be rounded to **2 decimals**.


## Rules
- Use the provided random seeds and hyperparameters.
- Do not change the grading variable names (Q1–Q4).
- Keep the model definitions as specified (do not tune beyond given parameters).


## 0. Setup

In [ ]:
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, ConfusionMatrixDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier

np.random.seed(0)


## 1. Download and load the dataset (do not modify)

We use UCI Letter Recognition:
- 20,000 samples
- 16 numeric features
- 26 classes (A–Z)

We follow the commonly-used split: **first 16,000 train**, **remaining 4,000 test**.

**Dataset: UCI Letter Recognition**

The UCI Letter Recognition dataset is a classical benchmark for multiclass classification, derived from optical character recognition (OCR) tasks. Each instance corresponds to a single uppercase English letter (A–Z), resulting in 26 classes. The dataset contains 20,000 samples, each represented by 16 numeric features that describe geometric and statistical properties of character images (such as stroke distributions, edge counts, and moment-based measures). Importantly, the original pixel images are not included; instead, the dataset provides hand-engineered feature representations extracted from images.

A commonly used experimental protocol is to train on the first 16,000 samples and evaluate on the remaining 4,000 samples, yielding a fixed and reproducible train–test split. Because the features are compact, low-dimensional, and informative, the dataset is well suited for studying the behavior of classical machine learning models. At the same time, the large number of classes and the visual similarity between many letters (e.g., C/G, O/Q, M/N) make the task non-trivial, which creates a natural setting to study error diversity and the benefits of ensemble methods such as voting and stacking.

In [ ]:
DATA_DIR = Path("letter_data")
DATA_DIR.mkdir(exist_ok=True)
DATA_PATH = DATA_DIR / "letter-recognition.data"

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/letter-recognition/letter-recognition.data"

if not DATA_PATH.exists():
    print("Downloading Letter Recognition dataset...")
    urllib.request.urlretrieve(URL, DATA_PATH)
    print("Download complete:", DATA_PATH)

raw = np.loadtxt(DATA_PATH, delimiter=",", dtype=str)

y = raw[:, 0]                 # 'A'..'Z'
X = raw[:, 1:].astype(float)  # 16 numeric features

# Fixed split (reproducible and simple for self-grading)
X_train, X_test = X[:16000], X[16000:]
y_train, y_test = y[:16000], y[16000:]

# Encode labels to 0..25 for convenience
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)
class_names = list(le.classes_)

X_train.shape, X_test.shape, len(class_names)


## 2. Evaluation helper (do not modify)

We compute:
- 5-fold stratified **CV macro F1** on the training set (parallelized with `n_jobs=-1`)
- **Test macro F1** and **Test accuracy** on the held-out test set


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def eval_model(model, X_train, y_train, X_test, y_test):
    # CV macro-F1 on the training set
    print("\t Cross-validation:")
    cv_f1 = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1, verbose=0).mean()
    print(f"\t\t Mean score: {cv_f1:.2f}")

    # Fit and evaluate on held-out test set
    print("\t Fitting the model...")
    model.fit(X_train, y_train)
    print("\t Getting the predictions...")
    pred = model.predict(X_test)

    print("\t Getting the scores...")
    test_f1 = f1_score(y_test, pred, average="macro")
    print(f"\t\t F1 on test: {test_f1:.2f}")
    test_acc = accuracy_score(y_test, pred)
    print(f"\t\t Acc on test: {test_acc:.2f}")

    return float(cv_f1), float(test_f1), float(test_acc)


## 3. Single models (baselines)

Build and evaluate the following three models.

### 3.1 Logistic Regression
Use a pipeline:
- `StandardScaler()`
- `LogisticRegression(max_iter=300, solver="lbfgs", random_state=0)`

### 3.2 Gaussian Naive Bayes
Use:
- `GaussianNB()` (no scaling required)

### 3.3 Decision Tree (regularized to avoid overfitting)
Use:
- `DecisionTreeClassifier(max_depth=6, min_samples_leaf=10, random_state=0)`

**Task:** Create the three models and compute their metrics using `eval_model`.

Store results in:
- `lr_cv_f1`, `lr_test_f1`, `lr_test_acc`
- `nb_cv_f1`, `nb_test_f1`, `nb_test_acc`
- `tree_cv_f1`, `tree_test_f1`, `tree_test_acc`


In [ ]:
# Write here your own code.
# Requirements:
# - define lr, nb, tree (exactly as specified above)
# - evaluate each using eval_model
# - define all 9 variables listed above
#
# YOUR CODE HERE


# SANITY CHECK (do not modify)

In [ ]:
# SANITY CHECK (do not modify)
required = [
    "lr_test_f1","nb_test_f1","tree_test_f1",
    "lr_cv_f1","nb_cv_f1","tree_cv_f1"
]
for r in required:
    assert r in globals(), f"{r} not defined"
for v in [lr_test_f1, nb_test_f1, tree_test_f1]:
    assert 0.0 <= v <= 1.0, "F1 should be in [0, 1]"
print("Sanity check passed: single-model scores look valid.")


### Question 1
Which **single model** achieved the highest **test macro F1**?

Set `Q1_best_single_model` to exactly one of:
- `"lr"`
- `"nb"`
- `"tree"`


In [ ]:
# ANSWER Q1
# Write your own code:
# - compare lr_test_f1, nb_test_f1, tree_test_f1
# - set Q1_best_single_model
#
# YOUR CODE HERE


## 4. Soft Voting Ensemble

Create a **soft voting** ensemble of the three base models (LR, NB, Tree).

Use:
- `VotingClassifier(..., voting="soft")`

**Task:** Define the voting model and compute:
- `vote_cv_f1`, `vote_test_f1`, `vote_test_acc`


In [ ]:
# Write here your own code.
# Requirements:
# - define voting (soft voting) using the three models (lr, nb, tree)
# - compute vote_cv_f1, vote_test_f1, vote_test_acc using eval_model
#
# YOUR CODE HERE


### Question 2
What is the **test macro F1** of the **soft voting** ensemble? (rounded to 2 decimals)

Define:
- `Q2_vote_test_f1`


In [ ]:
# ANSWER Q2
# Requirements:
# - use vote_test_f1 (computed above)
# - define Q2_vote_test_f1 as a float rounded to 2 decimals
#
# YOUR CODE HERE


## 5. Stacking Ensemble

Create a stacking model using the same base learners, and a logistic regression meta-learner.

Use:
- `final_estimator = LogisticRegression(max_iter=300, solver="lbfgs", random_state=0)`
- `passthrough = False`
- `stack_method = "predict_proba"`
- `cv = 3` (to keep runtime reasonable)

**Task:** Define the stacking model and compute:
- `stack_cv_f1`, `stack_test_f1`, `stack_test_acc`


In [ ]:
# Write here your own code.
# Requirements:
# - define stacking using lr, nb, tree as base estimators
# - final_estimator as specified above
# - stack_method="predict_proba"
# - passthrough=False
# - cv=3
# - compute stack_cv_f1, stack_test_f1, stack_test_acc using eval_model
#
# YOUR CODE HERE


### Question 3
What is the **test macro F1** of the **stacking** ensemble? (rounded to 2 decimals)

Define:
- `Q3_stack_test_f1`


In [ ]:
# ANSWER Q3
# Requirements:
# - use stack_test_f1 (computed above)
# - define Q3_stack_test_f1 as a float rounded to 2 decimals
#
# YOUR CODE HERE


## 6. Visualization

1. Confusion matrices on the **test set** for:
   - best single model
   - soft voting
   - stacking

2. Bar chart comparing **CV macro F1** across:
   - LR, NB, Tree, Voting, Stacking


In [ ]:
# Fit models for plotting (eval_model already fits them, but we refit explicitly for clarity)
lr.fit(X_train, y_train)
nb.fit(X_train, y_train)
tree.fit(X_train, y_train)
voting.fit(X_train, y_train)
stacking.fit(X_train, y_train)

single_models = {"lr": lr, "nb": nb, "tree": tree}
best_single = single_models[Q1_best_single_model]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
ConfusionMatrixDisplay.from_estimator(best_single, X_test, y_test, display_labels=class_names, ax=axes[0], xticks_rotation=90, cmap=None)
axes[0].set_title(f"Best single ({Q1_best_single_model})")

ConfusionMatrixDisplay.from_estimator(voting, X_test, y_test, display_labels=class_names, ax=axes[1], xticks_rotation=90, cmap=None)
axes[1].set_title("Soft voting")

ConfusionMatrixDisplay.from_estimator(stacking, X_test, y_test, display_labels=class_names, ax=axes[2], xticks_rotation=90, cmap=None)
axes[2].set_title("Stacking")

plt.tight_layout()
plt.show()

labels = ["lr","nb","tree","voting","stacking"]
cv_f1s = [lr_cv_f1, nb_cv_f1, tree_cv_f1, vote_cv_f1, stack_cv_f1]

plt.figure(figsize=(7,3))
plt.bar(labels, cv_f1s)
plt.ylabel("CV macro F1")
plt.title("5-fold CV macro F1 (training set)")
plt.tight_layout()
plt.show()


## Submission checklist

Your notebook must define:
- `Q1_best_single_model`
- `Q2_vote_test_f1`
- `Q3_stack_test_f1`
- `Q4_best_overall`
